# Code for the satisfaction final score

In [17]:
import pandas as pd
import numpy as np
import torch
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sentence_transformers import SentenceTransformer, util

/users/eleves-a/2022/adrien.bindel/ra_work/review-structure-analysis/venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/users/eleves-a/2022/adrien.bindel/ra_work/review-structure-analysis/venv/lib64/python3.9/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [57]:
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", cache_folder='../../final_pipeline/models/embedding_models').to(device)

In [ ]:
from langdetect import detect, DetectorFactory, LangDetectException

# Ensure consistent language detection results
DetectorFactory.seed = 0

def safe_detect(text):
    if not isinstance(text, str):
        return "unknown"
    
    # FIX: If text is very short, just assume English (or your main language)
    if len(text) < 15:
        return "en"
        
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"

def clean_data(df, review_col):

    df[review_col] = df[review_col].fillna("").astype(str)

    df['ID'] = df.index.astype(int)
    df['ID'] += 1  # Start IDs from 1 instead of 0

    df['lang'] = df[review_col].apply(safe_detect)

    return df

## Study 1

In [50]:
df1 = pd.read_excel('../../data/initial_data/Study 1 reviews.xlsx')

In [52]:
df1 = clean_data(df1, "finalReview")

df1.columns

Index(['ID', 'finalReview', 'Satisfaction_final', 'cleaning_service_quality',
       'order_packaging', 'communication_and_responsiveness',
       'Driver_professionalism', 'Service_speed',
       'cleaning_service_quality_sentiment', 'order_packaging_sentiment',
       'communication_and_responsiveness_sentiment',
       'Driver_professionalism_sentiment', 'Service_speed_sentiment',
       'Overall_review_sentiment', 'Emotional_intensity_LLM', 'lang'],
      dtype='object')

In [53]:
df_X = embedding_model.encode(
    df1["finalReview"].fillna("").tolist(), 
    # convert_to_tensor=True
    convert_to_tensor=False
)
df_y = df1["Satisfaction_final"]

In [54]:
df1.columns

Index(['ID', 'finalReview', 'Satisfaction_final', 'cleaning_service_quality',
       'order_packaging', 'communication_and_responsiveness',
       'Driver_professionalism', 'Service_speed',
       'cleaning_service_quality_sentiment', 'order_packaging_sentiment',
       'communication_and_responsiveness_sentiment',
       'Driver_professionalism_sentiment', 'Service_speed_sentiment',
       'Overall_review_sentiment', 'Emotional_intensity_LLM', 'lang'],
      dtype='object')

In [55]:
X = df_X
y = df_y.values

clf_1 = Ridge(random_state=42, alpha=1, solver="auto")

clf_1.fit(X, y)

y_pred = clf_1.predict(X)
y_pred_rounded = np.round(y_pred * 2) / 2

mae = mean_absolute_error(y,y_pred)
mae_rounded = mean_absolute_error(y, y_pred_rounded)

print(f"mae: {mae}")
print(f"mae with rounded predictions: {mae_rounded}")

mae: 0.8033246968362443
mae with rounded predictions: 0.7922751729438893


In [56]:
columns = list(df1.columns)

# columns = columns[1:]

columns = columns[:3] +  ["pred_Satisfaction_final"] + columns[3:]

df1["pred_Satisfaction_final"] = y_pred_rounded

df1 = df1.loc[:,columns]

df1.head()

,ID,finalReview,Satisfaction_final,pred_Satisfaction_final,cleaning_service_quality,order_packaging,communication_and_responsiveness,Driver_professionalism,Service_speed,cleaning_service_quality_sentiment,order_packaging_sentiment,communication_and_responsiveness_sentiment,Driver_professionalism_sentiment,Service_speed_sentiment,Overall_review_sentiment,Emotional_intensity_LLM,lang
0,1,My order was to dryclean! All suits came back ...,1.0,3.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en
1,2,poor experience. jacket not cleaned properly.,2.5,2.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en
2,3,The clean laundry came in a bag that had a sme...,4.5,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en
3,4,not happy with the service. i received multipl...,3.0,3.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en
4,5,its a very expensive service.,3.5,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en


In [ ]:
import pickle

# Save the model
with open('../../final_pipeline/models/satisfaction_final/ridge_model_study_1.pkl', 'wb') as f:
    pickle.dump(clf_1, f)

# To load it back later:
# with open('ridge_model.pkl', 'rb') as f:
#     loaded_model = pickle.load(f)

## Study 4

In [25]:
df4 = pd.read_excel('../../data/initial_data/Study 4 reviews.xlsx')

In [27]:
df4 = clean_data(df4, "text")

df4.columns

Index(['ID', 'text', 'Satisfaction_RA2', 'Quality_and_taste_of_food',
       'Cleanliness', 'Friendliness_of_staff', 'Value', 'Speed_of_service',
       'Quality_and_taste_of_food_sentiment', 'Cleanliness_sentiment',
       'Friendliness_of_staff_sentiment', 'Value_sentiment',
       'Speed_of_service_sentiment', 'Emotional_intensity_LLM', 'lang'],
      dtype='object')

In [28]:
df_X = embedding_model.encode(
    df4["text"].fillna("").tolist(), 
    # convert_to_tensor=True
    convert_to_tensor=False
)
df_y = df4["Satisfaction_RA2"]

In [29]:
X = df_X
y = df_y.values

clf_4 = Ridge(random_state=42, alpha=1, solver="auto")

clf_4.fit(X, y)

y_pred = clf_4.predict(X)
y_pred_rounded = np.round(y_pred * 2) / 2

mae = mean_absolute_error(y,y_pred)
mae_rounded = mean_absolute_error(y, y_pred_rounded)

print(f"mae: {mae}")
print(f"mae with rounded predictions: {mae_rounded}")

mae: 0.6390470862388611
mae with rounded predictions: 0.6364365816116333


In [30]:
df4.columns

Index(['ID', 'text', 'Satisfaction_RA2', 'Quality_and_taste_of_food',
       'Cleanliness', 'Friendliness_of_staff', 'Value', 'Speed_of_service',
       'Quality_and_taste_of_food_sentiment', 'Cleanliness_sentiment',
       'Friendliness_of_staff_sentiment', 'Value_sentiment',
       'Speed_of_service_sentiment', 'Emotional_intensity_LLM', 'lang'],
      dtype='object')

In [31]:
columns = list(df4.columns)

# columns = columns[1:]

columns = columns[:3] +  ["pred_Satisfaction_RA2"] + columns[3:]

df4["pred_Satisfaction_RA2"] = y_pred_rounded

df4 = df4.loc[:,columns]

df4.head()

,ID,text,Satisfaction_RA2,pred_Satisfaction_RA2,Quality_and_taste_of_food,Cleanliness,Friendliness_of_staff,Value,Speed_of_service,Quality_and_taste_of_food_sentiment,Cleanliness_sentiment,Friendliness_of_staff_sentiment,Value_sentiment,Speed_of_service_sentiment,Emotional_intensity_LLM,lang
0,1,Not so good,3,3.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en
1,2,Decent,7,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en
2,3,Could be better,4,3.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en
3,4,Alright,6,6.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en
4,5,OK,5,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en


In [32]:
import pickle

# Save the model
with open('../../final_pipeline/models/satisfaction_final/ridge_model_study_4.pkl', 'wb') as f:
    pickle.dump(clf_4, f)

# To load it back later:
# with open('ridge_model.pkl', 'rb') as f:
#     loaded_model = pickle.load(f)

## Study 5

In [35]:
df5 = pd.read_excel('../../data/initial_data/Study 5 reviews.xlsx')

In [36]:
df5.columns

Index(['ID', 'Review', 'Satisfaction_final', 'Cleanliness_and_maintenance',
       'Waiting_and_queuing_times', 'Quality_of_exhibits',
       'Quantity_of_exhibits', 'Helfpulness_of_staff',
       'Cleanliness_and_maintenance_sentiment',
       'Waiting_and_queuing_times_sentiment', 'Quality_of_exhibits_sentiment',
       'Quantity_of_exhibits_sentiment', 'Helfpulness_of_staff_sentiment',
       'Unnamed: 13', 'Emotional_intensity_LLM'],
      dtype='object')

In [38]:
df5 = clean_data(df5, "Review")

df5.columns

Index(['ID', 'Review', 'Satisfaction_final', 'Cleanliness_and_maintenance',
       'Waiting_and_queuing_times', 'Quality_of_exhibits',
       'Quantity_of_exhibits', 'Helfpulness_of_staff',
       'Cleanliness_and_maintenance_sentiment',
       'Waiting_and_queuing_times_sentiment', 'Quality_of_exhibits_sentiment',
       'Quantity_of_exhibits_sentiment', 'Helfpulness_of_staff_sentiment',
       'Unnamed: 13', 'Emotional_intensity_LLM', 'lang'],
      dtype='object')

In [39]:
df_X = embedding_model.encode(
    df5["Review"].fillna("").tolist(), 
    # convert_to_tensor=True
    convert_to_tensor=False
)
df_y = df5["Satisfaction_final"]

In [40]:
X = df_X
y = df_y.values

clf_5 = Ridge(random_state=42, alpha=1, solver="auto")

clf_5.fit(X, y)

y_pred = clf_5.predict(X)
y_pred_rounded = np.round(y_pred * 2) / 2

mae = mean_absolute_error(y,y_pred)
mae_rounded = mean_absolute_error(y, y_pred_rounded)

print(f"mae: {mae}")
print(f"mae with rounded predictions: {mae_rounded}")

mae: 0.5900683999061584
mae with rounded predictions: 0.5641562342643738


In [41]:
df5.columns

Index(['ID', 'Review', 'Satisfaction_final', 'Cleanliness_and_maintenance',
       'Waiting_and_queuing_times', 'Quality_of_exhibits',
       'Quantity_of_exhibits', 'Helfpulness_of_staff',
       'Cleanliness_and_maintenance_sentiment',
       'Waiting_and_queuing_times_sentiment', 'Quality_of_exhibits_sentiment',
       'Quantity_of_exhibits_sentiment', 'Helfpulness_of_staff_sentiment',
       'Unnamed: 13', 'Emotional_intensity_LLM', 'lang'],
      dtype='object')

In [42]:
columns = list(df5.columns)

# columns = columns[1:]

columns = columns[:3] +  ["pred_Satisfaction_final"] + columns[3:]

df5["pred_Satisfaction_final"] = y_pred_rounded

df5 = df5.loc[:,columns]

df5.head()

,ID,Review,Satisfaction_final,pred_Satisfaction_final,Cleanliness_and_maintenance,Waiting_and_queuing_times,Quality_of_exhibits,Quantity_of_exhibits,Helfpulness_of_staff,Cleanliness_and_maintenance_sentiment,Waiting_and_queuing_times_sentiment,Quality_of_exhibits_sentiment,Quantity_of_exhibits_sentiment,Helfpulness_of_staff_sentiment,Unnamed: 13,Emotional_intensity_LLM,lang
0,1,Very bad,1,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en
1,2,horrible,1,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en
2,3,Had a bad time here,2,2.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en
3,4,Crowded paths,5,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en
4,5,Horrible,1,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en


In [43]:
import pickle

# Save the model
with open('../../final_pipeline/models/satisfaction_final/ridge_model_study_5.pkl', 'wb') as f:
    pickle.dump(clf_5, f)

# To load it back later:
# with open('ridge_model.pkl', 'rb') as f:
#     loaded_model = pickle.load(f)